In [0]:
from datetime import timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T

# Update the production mapping history only after validating the proposed
# changes and checking that its Delta version has not changed since the read.
TARGET = "opsanalytics_adb_workspace01.capacity_modeling.service_groups"
CHARGE_SOURCE = "datahub_prod_ext_oci_dashbd_user.dashbd_user.oe_charge_detail"
spark.catalog.refreshTable(TARGET)
MAPPING_COLUMNS = [
    "LOC_NAME", "EPIC_DEPT_ID", "EXTERNAL_NAME", "SERVICE_GROUP", "VALID_FROM", "VALID_TO"
]
# Use one Spark run date for changed mappings and their VALID_TO dates.
# New departments instead start on their first qualifying bed service date.
# Remember the target's Delta version: if another run changes the table after
# this notebook reads it, the final cell stops instead of applying stale results.
run_date = spark.range(1).select(F.current_date().alias("run_date")).first().run_date
starting_version = DeltaTable.forName(spark, TARGET).history(1).select("version").first().version


In [0]:
# Read the complete mapping history and the current Oracle department reference.
# Read every row, including expired rows. We compare only active rows below, but
# Delta MERGE keeps the expired history already present in the target table.
current = spark.table(TARGET).select(*MAPPING_COLUMNS)

# Read the live Oracle reference each run. Secrets stay in Databricks; the
# notebook contains only secret names, never the URL or password values.
oracle_jdbc_url = dbutils.secrets.get(scope="oao_secrets", key="ORACLE_JDBC_URL")
oracle_password = dbutils.secrets.get(scope="oao_secrets", key="OAO_PRODUCTION")

# JDBC remains in use for the department reference. The bed-charge lookup
# below reads the live Oracle relation through its Databricks foreign catalog.
def oracle_table(dbtable):
    return (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("dbtable", dbtable)
        .option("user", "OAO_PRODUCTION")
        .option("password", oracle_password)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("oracle.net.ssl_server_dn_match", "true")
        .option("fetchsize", "10000")
        .load()
    )

# A trailing space or an empty string should not look like a changed mapping.
# Convert IDs to strings so Oracle and Delta can be joined even if their numeric
# column types differ. Empty values become null for consistent comparison.
def normalized(column):
    value = F.trim(F.col(column).cast("string"))
    return F.when(F.length(value) == 0, F.lit(None)).otherwise(value)

# These four Oracle fields define the latest desired mapping. DEPARTMENT_ID
# corresponds to EPIC_DEPT_ID in the Delta target. Other reference columns are
# intentionally ignored because the mapping table does not store them.
# Collect only the four department-mapping fields (a small reference table),
# then recreate a Spark DataFrame without Oracle JDBC in its lineage. Later
# actions, including Delta MERGE, will not open another Oracle connection.
clarity_rows = (
    oracle_table("(SELECT DEPARTMENT_ID, LOC_NAME, EXTERNAL_NAME, SERVICE_GROUP "
                 "FROM DASHBD_USER.CLARITY_DEP_REF) dept_ref")
    .select(
        normalized("DEPARTMENT_ID").alias("EPIC_DEPT_ID"),
        normalized("LOC_NAME").alias("LOC_NAME"),
        normalized("EXTERNAL_NAME").alias("EXTERNAL_NAME"),
        normalized("SERVICE_GROUP").alias("SERVICE_GROUP"),
    )
    .collect()
)
clarity_schema = T.StructType([
    T.StructField("EPIC_DEPT_ID", T.StringType(), True),
    T.StructField("LOC_NAME", T.StringType(), True),
    T.StructField("EXTERNAL_NAME", T.StringType(), True),
    T.StructField("SERVICE_GROUP", T.StringType(), True),
])
clarity_dep_ref = spark.createDataFrame(clarity_rows, clarity_schema)

# A missing/duplicated reference ID would make the update ambiguous. Stop
# before any write so the job cannot silently choose one of two mappings.
if clarity_dep_ref.count() == 0:
    raise ValueError("Clarity department reference is empty")
if clarity_dep_ref.groupBy("EPIC_DEPT_ID").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Clarity department reference has duplicate department IDs")

# VALID_TO = null marks the current mapping for a department. Historical
# rows with a VALID_TO date remain in the table and do not take part in the join.
active = current.filter(F.col("VALID_TO").isNull()).withColumn(
    "EPIC_DEPT_ID", normalized("EPIC_DEPT_ID")
)
if active.filter(F.col("EPIC_DEPT_ID").isNull()).limit(1).count():
    raise ValueError("Active service group mapping has a blank department ID")
if active.groupBy("EPIC_DEPT_ID").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Active service group mapping has duplicate department IDs")


In [0]:
# Compare active, non-emergency mappings. Missing reference rows do not retire mappings.
# Intent check: preserve the legacy rule that existing departments whose
# current EXTERNAL_NAME contains EMERGENCY are not updated by this process.
# They still count as existing IDs, so they will not be mistaken for new ones.
tracked = active.filter(
    ~F.upper(F.coalesce(F.col("EXTERNAL_NAME"), F.lit(""))).contains("EMERGENCY")
)
# Give current and Oracle values distinct names so the comparison is explicit.
# VALID_FROM is retained to decide later whether to close history or replace a
# row that was already created today.
old = tracked.select(
    "EPIC_DEPT_ID",
    normalized("LOC_NAME").alias("old_loc"),
    normalized("EXTERNAL_NAME").alias("old_name"),
    normalized("SERVICE_GROUP").alias("old_group"),
    F.col("VALID_FROM").cast("date").alias("old_valid_from"),
)
ref = clarity_dep_ref.select(
    "EPIC_DEPT_ID",
    F.col("LOC_NAME").alias("new_loc"),
    F.col("EXTERNAL_NAME").alias("new_name"),
    F.col("SERVICE_GROUP").alias("new_group"),
)
# Inner join = IDs present in both active mappings and Oracle. Compare only
# location, department name, and service group, after trimming whitespace.
# Null-safe equality treats two missing values as equal.
changes = (
    old.join(ref, "EPIC_DEPT_ID", "inner")
    .filter(
        ~F.col("old_loc").eqNullSafe(F.col("new_loc"))
        | ~F.col("old_name").eqNullSafe(F.col("new_name"))
        | ~F.col("old_group").eqNullSafe(F.col("new_group"))
    )

)

# Closing a row with an unknown or future start date would corrupt its
# validity interval, so fail before preparing any write.
if changes.filter(F.col("old_valid_from").isNull() | (F.col("old_valid_from") > F.lit(run_date))).limit(1).count():
    raise ValueError("A changed mapping has a missing or future VALID_FROM date")



# Show the proposed old/new values before the write cell. These are
# department-level mapping fields only, not charge-detail records.
print(f"Changed mappings: {changes.count()}")
display(changes.select("EPIC_DEPT_ID", "old_loc", "new_loc", "old_name", "new_name", "old_group", "new_group"))


In [0]:
# Left anti join = reference IDs with no active target mapping. A department
# with only expired history is eligible to be re-added. An active ID missing
# from Oracle is deliberately left unchanged; this process does not retire it.
# Intent check: a new EMERGENCY department can still qualify through bed charges.
new_candidates = clarity_dep_ref.join(active.select("EPIC_DEPT_ID"), "EPIC_DEPT_ID", "left_anti")
# Show all missing active IDs first. The charge check below is a separate
# eligibility filter, so its displayed count can be smaller than this count.
print(f"New department candidates: {new_candidates.count()}")
display(new_candidates)

# Read the live charge relation through its Databricks foreign catalog.
# Restrict the scan to new department IDs and collect only department-level
# aggregates, never patient-level charge rows. The first qualifying bed
# service date becomes the new mapping's VALID_FROM date.
candidate_ids = [row.EPIC_DEPT_ID for row in new_candidates.select("EPIC_DEPT_ID").collect()]
if any(not department_id.isascii() or not department_id.isdigit() for department_id in candidate_ids):
    raise ValueError("A new department ID is not numeric")

# Keep the legacy 2024-06-01 threshold and exact BED CHARGES category.
# The foreign relation should expose Oracle's VARCHAR2(8) YYYYMMDD dates.
bed_date_schema = T.StructType([
    T.StructField("EPIC_DEPT_ID", T.StringType(), False),
    T.StructField("first_bed_service_date", T.DateType(), True),
])
if candidate_ids:
    charge = spark.table(CHARGE_SOURCE)
    required_columns = {"EPIC_DEPT_ID", "BILLING_CAT_DESC", "SERVICE_DATE"}
    missing_columns = required_columns - set(charge.columns)
    if missing_columns:
        raise ValueError(f"Foreign charge table is missing columns: {sorted(missing_columns)}")
    if not isinstance(charge.schema["SERVICE_DATE"].dataType, (T.StringType, T.VarcharType)):
        raise TypeError("Foreign charge SERVICE_DATE must be a YYYYMMDD string or varchar")

    # Apply candidate and date filters before grouping for possible pushdown.
    # Check the date format in the same aggregate that finds the first date.
    charge_aggregates = (
        charge.select("EPIC_DEPT_ID", "BILLING_CAT_DESC", "SERVICE_DATE")
        .filter(F.col("EPIC_DEPT_ID").isin(candidate_ids))
        .filter(F.col("BILLING_CAT_DESC") == "BED CHARGES")
        .filter(F.col("SERVICE_DATE") >= "20240601")
        .groupBy("EPIC_DEPT_ID")
        .agg(
            F.min("SERVICE_DATE").alias("FIRST_SERVICE_DATE"),
            F.max(
                F.when(~F.col("SERVICE_DATE").rlike(r"^[0-9]{8}$"), 1).otherwise(0)
            ).alias("INVALID_SERVICE_DATE_FORMAT"),
        )
        .select(
            normalized("EPIC_DEPT_ID").alias("EPIC_DEPT_ID"),
            F.to_date("FIRST_SERVICE_DATE", "yyyyMMdd").alias("first_bed_service_date"),
            "INVALID_SERVICE_DATE_FORMAT",
        )
        .collect()
    )
    if any(row.INVALID_SERVICE_DATE_FORMAT for row in charge_aggregates):
        raise ValueError("A qualifying bed charge has a non-YYYYMMDD SERVICE_DATE")
    bed_date_rows = [
        (row.EPIC_DEPT_ID, row.first_bed_service_date) for row in charge_aggregates
    ]
else:
    bed_date_rows = []
bed_dates = spark.createDataFrame(bed_date_rows, bed_date_schema)

# Keep the Oracle date alongside each matching Clarity mapping. Departments
# without a qualifying bed charge remain candidates but are not inserted.
new_with_bed_charges = new_candidates.join(F.broadcast(bed_dates), "EPIC_DEPT_ID", "inner")
if new_with_bed_charges.filter(F.col("first_bed_service_date").isNull()).limit(1).count():
    raise ValueError("A qualifying bed charge has an invalid SERVICE_DATE")
print(f"New departments with bed charges: {new_with_bed_charges.count()}")
display(new_with_bed_charges)

In [0]:
# Validate the rows that would be written. Keep all historical rows in the target.
# Build the exact mappings we would insert/update, using the Oracle values
# for changed rows and the reference values for newly qualified departments.
# No target writes have happened yet.
changed_rows = changes.select(
    "EPIC_DEPT_ID",
    F.col("new_loc").alias("LOC_NAME"),
    F.col("new_name").alias("EXTERNAL_NAME"),
    F.col("new_group").alias("SERVICE_GROUP"),
    "old_valid_from",
    F.lit(run_date).alias("valid_from"),
)
new_rows = new_with_bed_charges.select(
    "EPIC_DEPT_ID", "LOC_NAME", "EXTERNAL_NAME", "SERVICE_GROUP",
    F.lit(None).cast("date").alias("old_valid_from"),
    F.col("first_bed_service_date").alias("valid_from"),
)
# Intent check: a missing service group (or name/location) blocks the whole
# update rather than creating a partially classified active mapping.
to_write = changed_rows.unionByName(new_rows)
if to_write.filter(
    F.col("LOC_NAME").isNull()
    | F.col("EXTERNAL_NAME").isNull()
    | F.col("SERVICE_GROUP").isNull()
).limit(1).count():
    raise ValueError("A mapping to write is missing a location, department name, or service group")

# Date-granularity history cannot close a row before its start. Update same-day
# changes in place; close older rows yesterday and insert today's replacement.
# A change to a row started before today gets two actions: expire that
# active row yesterday, then insert the new mapping starting today. A second
# change on the same day updates today's row in place, because yesterday would
# be earlier than its VALID_FROM date.
older_changes = changed_rows.filter(F.col("old_valid_from") < F.lit(run_date))
same_day_changes = changed_rows.filter(F.col("old_valid_from") == F.lit(run_date))

# Stage 'close' and 'replace' rows with a merge key so they match the active
# target row. Stage 'insert' rows with a null merge key so MERGE treats them as
# new rows, even when they share an ID with a row being expired in this run.
def stage(frame, operation, merge_key):
    return frame.select(
        F.lit(operation).alias("operation"),
        (F.col("EPIC_DEPT_ID") if merge_key else F.lit(None).cast("string")).alias("merge_key"),
        "EPIC_DEPT_ID", "LOC_NAME", "EXTERNAL_NAME", "SERVICE_GROUP", "valid_from",
    )

staged = (
    stage(older_changes, "close", True)
    .unionByName(stage(same_day_changes, "replace", True))
    .unionByName(stage(older_changes, "insert", False))
    .unionByName(stage(new_rows, "insert", False))
)

# No differences means no Delta write or new table version. On a write,
# recheck the version captured in cell 1; rerun all cells if the table changed.
# The MERGE applies history closure and insertion as one Delta transaction.
if to_write.limit(1).count():
    target = DeltaTable.forName(spark, TARGET)
    latest_version = target.history(1).select("version").first().version
    if latest_version != starting_version:
        raise RuntimeError("Service group table changed since this notebook read it; rerun all cells")
    (
        target.alias("t")
        .merge(
            staged.alias("s"),
            "CAST(t.EPIC_DEPT_ID AS STRING) = s.merge_key AND t.VALID_TO IS NULL",
        )
        .whenMatchedUpdate(
            condition="s.operation = 'close'",
            set={"VALID_TO": F.lit(run_date - timedelta(days=1))},
        )
        .whenMatchedUpdate(
            condition="s.operation = 'replace'",
            set={
                "LOC_NAME": F.col("s.LOC_NAME"),
                "EXTERNAL_NAME": F.col("s.EXTERNAL_NAME"),
                "SERVICE_GROUP": F.col("s.SERVICE_GROUP"),
            },
        )
        # Changed rows start today; new IDs start on their first bed service date.
        # Unchanged departments and expired historical rows are untouched.
        .whenNotMatchedInsert(
            condition="s.operation = 'insert'",
            values={
                "LOC_NAME": F.col("s.LOC_NAME"),
                "EPIC_DEPT_ID": F.col("s.EPIC_DEPT_ID"),
                "EXTERNAL_NAME": F.col("s.EXTERNAL_NAME"),
                "SERVICE_GROUP": F.col("s.SERVICE_GROUP"),
                "VALID_FROM": F.col("s.valid_from"),
                "VALID_TO": F.lit(None).cast(current.schema["VALID_TO"].dataType),
            },
        )
        .execute()
    )
    print("Service group mappings updated")
else:
    print("No service group mappings to update")
